In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [3]:
vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming!"], embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4o-mini")

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("What is Teddy's occupation?")

"Teddy's occupation is an AI engineer."

In [4]:
from operator import itemgetter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [5]:
vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming!"],
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

chain.invoke({"question": "What is Teddyy's occupation?", "language": "Korean"})

'테디의 직업은 AI 엔지니어입니다.'

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

In [8]:
model = ChatOpenAI()
capital_chain = (
    ChatPromptTemplate.from_template("{country}의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

area_chain = (
    ChatPromptTemplate.from_template("{country}의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

map_chain = RunnableParallel(capital=capital_chain, are=area_chain)

map_chain.invoke({"country": "대한민국"})

{'capital': '대한민국의 수도는 서울입니다.', 'are': '대한민국의 총 면적은 약 100,363제곱 킬로미터 입니다.'}